# 01 · Knowledge, RAG and Foundry IQ

> **The story:** Tenants email DLD every day asking whether their landlord's rent increase is allowed. The answer depends on Dubai's rules: how far the rent is below the area average, and whether the landlord gave 90 days' notice. Sara, a DLD rental-services officer, gets an AI team to read the email, check the rules, do the maths, draft the reply and book a mediation appointment, and nothing is sent or booked until she approves.

**You will learn**
- Why an AI without your documents sounds confident but guesses
- Classic RAG: one search over an index you build yourself
- Foundry IQ: Foundry ingests the PDFs itself, plans several searches, and cites sources
- Five controls against made-up answers: confidence scores, strict instructions, a hallucination catcher, citation checks and conversation memory

**Time:** about 30 minutes · **Before you start:**
- Notebook 00 finished successfully

<sub>Workshop material: the rental documents summarise Dubai rules for training only. Not legal advice.</sub>

In [ ]:
# @title Setup: run this first (about 1 minute, then sign in)
from pathlib import Path
import os, shutil, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # The workshop code goes on this notebook's temporary Colab machine (a few seconds; nothing in Drive).
    ROOT = Path("/content/dld-foundry-workshop")
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=False)   # pick up any fixes
    else:
        subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/NetwaysPresales/dld-foundry-workshop", str(ROOT)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)])
else:
    here = Path.cwd().resolve()
    ROOT = next((p for p in [here, *here.parents] if (p / "dld_workshop").exists()), here)
sys.path.insert(0, str(ROOT))

from azure.identity import AzureCliCredential, DeviceCodeCredential
from dld_workshop import WorkshopEnvironment, WorkshopSettings
from dld_workshop.display import show, table, checklist

# In Colab you'll see a link and a code: open the link, enter the code, sign in with your work account.
credential = DeviceCodeCredential() if IN_COLAB else AzureCliCredential()

settings = WorkshopEnvironment.discover(credential)
print(f"✅ Signed in. Your environment: {settings.resource_group}")

## Step 1 · The rulebook

Six short documents: three summarise real Dubai rules (Decree 43 of 2013 rent caps, the 90-day notice in Law 26 of 2007, the Rental Disputes Centre), three are workshop procedures (how to reply, when to book mediation, data handling).

In [ ]:
pdf_dir = ROOT / "workshop" / "knowledge_sources" / "rental_service"
table([{"document": p.name} for p in sorted(pdf_dir.glob("*.pdf"))])

## Step 2 · Load the rulebook two ways

- **Classic RAG:** we cut the text into passages, create embeddings and fill a search index ourselves.
- **Foundry IQ:** we only point it at the PDFs in Blob Storage; Search extracts, chunks, embeds and indexes them itself.

Takes about 2 to 4 minutes.

In [ ]:
from dld_workshop.knowledge import upload_policy_documents, create_classic_rag_index, create_foundry_iq, wait_for_foundry_iq

uploaded = upload_policy_documents(settings, credential, pdf_dir)
passages = create_classic_rag_index(settings, credential, pdf_dir / "source")
create_foundry_iq(settings, credential)
status = wait_for_foundry_iq(settings, credential)
print(f"✅ {len(uploaded)} PDFs in Blob Storage")
print(f"✅ Classic RAG index: {passages} passages we built ourselves")
print(f"✅ Foundry IQ ingested {status['documents_processed']} documents on its own")

## Step 3 · One question, three ways

In [ ]:
from dld_workshop.rag import no_rag, classic_rag, foundry_iq

QUESTION = """
A tenant in Dubai pays AED 80,000 a year. The RERA index average for similar units is AED 100,000.
The landlord wants a 15% increase at renewal and told the tenant by WhatsApp last week; the contract
ends in 6 weeks. Is the increase allowed, is the notice valid, and what can the tenant do?
Cite the sources, and say so if something isn't covered.
""".strip()

results = [no_rag(settings, credential, QUESTION),
           classic_rag(settings, credential, QUESTION),
           foundry_iq(settings, credential, QUESTION)]
table(results, ["pattern", "sources", "searches", "tokens", "seconds"])

In [ ]:
for result in results:
    show(result["pattern"], result["answer"])
print("Searches Foundry IQ planned on its own:")
for query in results[2]["search_queries"]:
    print("  •", query)

> ### 👉 Follow along in the portal
>
> Foundry portal → your project → **Build → Knowledge**: the knowledge base `dld-rental-rules-knowledge-base` and its knowledge source `dld-rental-rules-blob-source` are there, created by the code you just ran.
> Open the knowledge source to see the Blob container, the embedding model and the ingestion status. **Create knowledge base → Azure Blob Storage** in the portal builds exactly the same thing.

# Part 2 · Can we trust the answer?
Grounding reduces made-up answers but doesn't eliminate them. Five controls, each shown working on real questions:

| # | Control | What it stops |
|---|---|---|
| 4 | **Confidence scores** | Answering when the documents don't really match |
| 5 | **Strict instructions** | The model filling gaps with plausible inventions |
| 6 | **Hallucination catcher** (groundedness detection) | Sentences the sources don't support slipping through |
| 7 | **Citation check** | Citing a document that was never retrieved |
| 8 | **Conversation memory** | Follow-up questions losing what they refer to |

## Step 4 · Confidence scores

Search's reranker scores every passage from **0** (unrelated) to **4** (highly relevant). The model may only answer from passages scoring **1.8 or more**; below that we refuse **without calling the model at all**.

Three questions: one the rulebook covers, one completely off-topic, and one that's *related* but not actually answered.

In [ ]:
from dld_workshop.grounding import answer_with_confidence

QUESTIONS = [
    "My rent is 20% below the RERA index average. What is the maximum increase my landlord can ask for at renewal?",
    "What are the opening hours of the DLD customer happiness centre in Deira?",
    "How much compensation can a tenant claim if the landlord raised the rent unlawfully?",
]
checks = [answer_with_confidence(settings, credential, q, threshold=1.8) for q in QUESTIONS]
table(checks, ["question", "best_score", "confidence", "passages_used", "answered"])
for check in checks:
    show(f"{check['confidence']} confidence · {check['question']}", check["answer"])

In [ ]:
# The scores behind the first and second question
table(checks[0]["passages"], ["score", "source_file", "preview"])
table(checks[1]["passages"], ["score", "source_file", "preview"])

## Step 5 · Strict vs. loose instructions

Both answers get **the same passages**. The loose prompt is a realistic, badly written one: *"always give a specific amount and time, never say you don't know."* The rulebook says nothing about compensation or how long a dispute takes.

Run it twice: models vary, and the loose prompt often invents an amount or a timeline.

In [ ]:
from dld_workshop.grounding import strict_vs_loose

PRESSURE_QUESTION = ("My landlord raised my rent unlawfully. How much compensation will I get, and how many "
                     "days will the Rental Disputes Centre take to decide?")
versions = strict_vs_loose(settings, credential, PRESSURE_QUESTION)
show("Loose instructions", versions["loose"])
show("Strict instructions", versions["strict"])

## Step 6 · Hallucination catcher

**Groundedness detection** (Azure AI Content Safety, built into your Foundry resource) compares an answer with its sources and lists every sentence they don't support. It checks both answers from step 5, plus a good answer with one invented sentence added on purpose.

In [ ]:
from dld_workshop.grounding import detect_groundedness

GOOD_QUESTION = QUESTIONS[0]
good_answer = checks[0]["answer"]
planted = good_answer + " Tenants are also entitled to an automatic AED 5,000 refund if the landlord's notice is late."
good_sources = [p["content"] for p in checks[0]["passages"]]

rows = []
for label, question, answer, sources in [
    ("Good answer (step 4)", GOOD_QUESTION, good_answer, good_sources),
    ("Good answer + 1 invented sentence", GOOD_QUESTION, planted, good_sources),
    ("Loose answer (step 5)", PRESSURE_QUESTION, versions["loose"], versions["sources"]),
    ("Strict answer (step 5)", PRESSURE_QUESTION, versions["strict"], versions["sources"]),
]:
    result = detect_groundedness(settings, credential, question=question, answer=answer, sources=sources)
    rows.append({"answer": label, "ungrounded?": "🚩 yes" if result["ungrounded"] else "✅ no",
                 "% unsupported": result["ungrounded_percentage"],
                 "flagged sentences": " | ".join(result["ungrounded_sentences"])[:300]})
table(rows)

## Step 7 · Citation check

A citation is only worth something if the cited document was actually retrieved. We compare what the answer cites with the passages behind it, then add a fake citation.

In [ ]:
from dld_workshop.grounding import verify_citations

real = verify_citations(good_answer, checks[0]["passages"])
fake = verify_citations(good_answer + " (07_rent_refunds.pdf)", checks[0]["passages"])
table([
    {"answer": "Real answer", "cited": ", ".join(real["cited"]), "not in evidence": ", ".join(real["not_in_evidence"]) or "-", "ok": real["ok"]},
    {"answer": "With a fake citation", "cited": ", ".join(fake["cited"]), "not in evidence": ", ".join(fake["not_in_evidence"]) or "-", "ok": fake["ok"]},
])

## Step 8 · Conversation memory

Sara's follow-up is *"And how much notice must he give, and what can I do if he refuses?"* On its own, "he" could be anyone. Foundry IQ plans its searches from the **whole conversation**: compare the searches and sources with and without the earlier turns.

In [ ]:
from dld_workshop.grounding import foundry_iq_turn

TURN_1 = "My landlord wants to raise my rent by 15% at renewal. My rent is 20% below the RERA index average."
REPLY_1 = "In that band the maximum increase at renewal is 5% of the current rent."
TURN_2 = "And how much notice must he give me, and what can I do if he refuses to back down?"

alone = foundry_iq_turn(settings, credential, [("user", TURN_2)])
remembered = foundry_iq_turn(settings, credential, [("user", TURN_1), ("assistant", REPLY_1), ("user", TURN_2)])

for label, result in [("Follow-up alone", alone), ("Follow-up with the conversation", remembered)]:
    print(f"\n{label}: searches Foundry IQ planned")
    for search in result["searches"]:
        print("  •", search)
    table(result["references"])
show("Answer with the conversation", remembered["answer"])

Foundry IQ can also drop weak passages itself with a **reranker threshold**. Watch the number of sources fall as the bar rises:

In [ ]:
table([{"reranker threshold": t,
        "sources kept": len(foundry_iq_turn(settings, credential, [("user", TURN_1)], reranker_threshold=t)["references"])}
       for t in [None, 2.0, 2.5, 3.0]])

> ### 👉 Follow along in the portal
>
> **Build → Knowledge → `dld-rental-rules-knowledge-base`** → open the test playground: ask TURN_1, then TURN_2 in the same chat, and open the activity panel to see the searches it planned.
> **Guardrails → Create guardrail**: groundedness is one of the controls you can switch on for a model or agent, the platform version of step 6.

---
## What to notice
- A low confidence score stops a bad answer before the model even runs.
- Instructions matter: the same evidence can produce an invented amount under pressure, and 'not covered' under strict rules.
- Groundedness detection names the exact unsupported sentence; the citation check catches a fake source.
- Follow-ups only work if the whole conversation is used to plan the search.

**Try it yourself:** write your own invented sentence into `planted` and see whether the catcher flags it.